<a href="https://colab.research.google.com/github/kumarmaheshbly-oss/Project-Aura-/blob/main/eckstein_hechler_propagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install "orekit-jpype[jdk4py]" --quiet

In [4]:

# 1. Packages install karein (agar pehle se hain toh skip ho jayega)
!pip install "orekit-jpype[jdk4py]" -q

import os
import jdk4py
import jpype
import orekit_jpype

# 2. Java path set karein
os.environ["JAVA_HOME"] = str(jdk4py.JAVA_HOME)

# 3. JVM start karein
if not jpype.isJVMStarted():
    orekit_jar_dir = os.path.join(os.path.dirname(orekit_jpype.__file__), "jars")
    orekit_jar_path = os.path.join(orekit_jar_dir, "*")
    jpype.startJVM(classpath=[orekit_jar_path])

# 4. Orekit data setup karein
from orekit_jpype.pyhelpers import download_orekit_data_curdir, setup_orekit_curdir
download_orekit_data_curdir()
setup_orekit_curdir()

print("✅ JVM aur Orekit Data Ready! Ab aap agla cell chala sakte hain.")

✅ JVM aur Orekit Data Ready! Ab aap agla cell chala sakte hain.


In [7]:

from org.orekit.propagation.analytical import EcksteinHechlerPropagator
from org.orekit.attitudes import FrameAlignedProvider
from org.orekit.frames import FramesFactory
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.utils import Constants
from org.orekit.orbits import KeplerianOrbit, PositionAngleType

# 1. Time set karein (20 March 2024, 12:00 UTC)
date = AbsoluteDate(2024, 3, 20, 12, 0, 0.0, TimeScalesFactory.getUTC())

# 2. Initial Orbit banayein (ISS jaisa 550 km altitude aur 51.6° inclination)
a = Constants.WGS84_EARTH_EQUATORIAL_RADIUS + 550000.0
e = 0.0001
i = 0.9006
omega = 0.0
raan = 0.0
lv = 0.0

initial_orbit = KeplerianOrbit(a, e, i, omega, raan, lv,
                               PositionAngleType.TRUE,
                               FramesFactory.getEME2000(),
                               date,
                               Constants.WGS84_EARTH_MU)

# 3. Attitude Provider (Satellite ki orientation)
attitude_provider = FrameAlignedProvider.of(initial_orbit.getFrame())

# 4. Eckstein-Hechler Propagator setup karein
eh_propagator = EcksteinHechlerPropagator(
    initial_orbit,
    attitude_provider,
    400.0,                                  # Satellite Mass (kg)
    Constants.WGS84_EARTH_EQUATORIAL_RADIUS,  # Reference Radius
    Constants.WGS84_EARTH_MU,                 # Gravitational Parameter
    Constants.WGS84_EARTH_C20,                # J2 coefficient
    0.0,                                      # c30
    0.0,                                      # c40
    0.0,                                      # c50
    0.0                                       # c60
)

# 5. 1 ghante baad ka time calculate karein
date_plus_1hr = date.shiftedBy(3600.0)
final_state_eh = eh_propagator.propagate(date_plus_1hr)
new_pos_eh = final_state_eh.getPVCoordinates().getPosition()

print("🛰️ Eckstein-Hechler: 1 Ghante Baad Position in km:")
print(f"X: {new_pos_eh.getX() / 1000:.2f}")
print(f"Y: {new_pos_eh.getY() / 1000:.2f}")
print(f"Z: {new_pos_eh.getZ() / 1000:.2f}")

🛰️ Eckstein-Hechler: 1 Ghante Baad Position in km:
X: -4786.34
Y: -3095.32
Z: -3920.42
